In [1]:
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [5]:
def extract_nasa_features(mat_file_path):
    mat = scipy.io.loadmat(mat_file_path)
    # The key matches the filename (e.g., 'B0005')
    filename = list(mat.keys())[-1] 
    cycles = mat[filename][0, 0]['cycle'][0]
    
    cycle_data = []
    
    for c in cycles:
        if c['type'][0] == 'discharge':
            data = c['data'][0, 0]
            
            # Ground Truth Capacity for this cycle
            true_capacity = data['Capacity'][0, 0]
            
            # Time series array vectors for this discharge cycle
            v_array = data['Voltage_measured'][0]
            i_array = data['Current_measured'][0]
            t_array = data['Time'][0]
            temp_array = data['Temperature_measured'][0]
            
            # Your Feature Idea: Calculate cumulative discharge capacity (Ah) 
            # strictly within a specific high-voltage window (e.g., 4.0V down to 3.2V)
            cutoff_indices = np.where((v_array <= 4.0) & (v_array >= 3.2))[0]
            
            if len(cutoff_indices) > 1:
                v_trunc = v_array[cutoff_indices]
                i_trunc = np.abs(i_array[cutoff_indices])
                t_trunc = t_array[cutoff_indices]
                
                # Trapezius integration of current over time (seconds -> hours)
                dt = np.diff(t_trunc)
                avg_i = (i_trunc[:-1] + i_trunc[1:]) / 2
                voltage_window_cap = np.sum(avg_i * dt) / 3600.0
            else:
                voltage_window_cap = 0.0
                
            # Extra engineered cycle-level features
            initial_temp = temp_array[0]
            max_temp = np.max(temp_array)
            total_discharge_time = t_array[-1] - t_array[0]
            
            cycle_data.append({
                'window_capacity_ah': voltage_window_cap,
                'initial_temp': initial_temp,
                'max_temp': max_temp,
                'duration': total_discharge_time,
                'target_capacity': true_capacity
            })
            
    return pd.DataFrame(cycle_data)

# Example usage:
# df_b0005 = extract_nasa_features('B0005.mat')


In [6]:

class BatterySequenceDataset(Dataset):
    def __init__(self, df, lookback=5):
        # Separate features and target
        features = df[['window_capacity_ah', 'initial_temp', 'max_temp', 'duration']].values
        targets = df['target_capacity'].values
        
        # Scale features for neural networks
        self.scaler = StandardScaler()
        scaled_features = self.scaler.fit_transform(features)
        
        self.X = []
        self.y = []
        
        # Create sliding windows
        for i in range(len(scaled_features) - lookback):
            self.X.append(scaled_features[i : i + lookback])
            self.y.append(targets[i + lookback])
            
        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32).unsqueeze(-1)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    


In [7]:

class BatterySohGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BatterySohGRU, self).__init__()
        self.gru = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1) # Outputs a single value: Capacity
        
    def forward(self, x):
        # out shape: [batch_size, seq_len, hidden_dim]
        out, _ = self.gru(x)
        # Take the hidden state of the very last element in the sequence
        last_hidden = out[:, -1, :]
        # Predict capacity
        prediction = self.fc(last_hidden)
        return prediction
    


In [15]:

# 1. Load and process data from files
df_train1 = extract_nasa_features('Data/B0005.mat')
df_train2 = extract_nasa_features('Data/B0006.mat')
df_train = pd.concat([df_train1, df_train2], ignore_index=True)
df_test = extract_nasa_features('Data/B0007.mat')


In [20]:
df_train1.head()

,window_capacity_ah,initial_temp,max_temp,duration,target_capacity
0,1.841141,24.330034,38.982181,3690.234,1.856487
1,1.825390,24.697752,39.033398,3672.344,1.846327
2,1.814064,24.734266,38.818797,3651.641,1.835349
3,1.819532,24.654236,38.762305,3631.563,1.835263
4,1.818851,24.524797,38.665393,3629.172,1.834646


In [16]:

# 2. Convert to sequences
lookback_length = 5
train_dataset = BatterySequenceDataset(df_train, lookback=lookback_length)
test_dataset = BatterySequenceDataset(df_test, lookback=lookback_length)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)


In [17]:


# 3. Initialize Model, Loss, and Optimizer
model = BatterySohGRU(input_dim=4, hidden_dim=16, num_layers=1)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [18]:
# 4. Training Loop
epochs = 50
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Training MSE Loss: {epoch_loss/len(train_loader):.5f}")


Epoch 10/50 | Training MSE Loss: 0.00740
Epoch 20/50 | Training MSE Loss: 0.00383
Epoch 30/50 | Training MSE Loss: 0.00287
Epoch 40/50 | Training MSE Loss: 0.00260
Epoch 50/50 | Training MSE Loss: 0.00233


In [19]:

# 5. Evaluate on Unseen Battery Cell (B0007)
model.eval()
with torch.no_grad():
    test_X, test_y = test_dataset.X, test_dataset.y
    test_preds = model(test_X)
    test_loss = criterion(test_preds, test_y)
    print(f"\nEvaluation on Battery B0007 - Mean Squared Error: {test_loss.item():.5f}")


Evaluation on Battery B0007 - Mean Squared Error: 0.01113
